# Model 5: PyTorch LSTM Deep Learning Model for Drug `M01AE`

## Hyperparameter Selection Methodology:
We evaluate sequence lookback length (28 days), hidden dimensions (64), and learning rates $\{0.001, 0.005, 0.01\}$ on validation set performance.


In [1]:
# Dynamic Dependency Guard & Environment Initialization
import sys, subprocess, os

def install_and_import(pkg, module_name=None):
    if module_name is None:
        module_name = pkg
    try:
        __import__(module_name)
    except ImportError:
        print(f"Installing missing dependency: {pkg}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

install_and_import('numpy')
install_and_import('pandas')
install_and_import('matplotlib')
install_and_import('seaborn')
install_and_import('scikit-learn', 'sklearn')
install_and_import('statsmodels')
install_and_import('lightgbm')
install_and_import('xgboost')
install_and_import('shap')
install_and_import('prophet')
install_and_import('optuna')
install_and_import('torch')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style="whitegrid")
plt.rcParams.update({
    'font.sans-serif': 'Inter, Roboto, Arial, sans-serif',
    'axes.edgecolor': '#cccccc',
    'axes.linewidth': 1.0,
    'grid.color': '#eeeeee',
    'grid.linestyle': '--'
})

TARGET_DRUG = 'M01AE'
data_dir = r'c:\Users\ranje\sales forcasting\times_series\dataset'

train_df = pd.read_csv(os.path.join(data_dir, 'train_daily.csv'))
val_df   = pd.read_csv(os.path.join(data_dir, 'val_daily.csv'))
test_df  = pd.read_csv(os.path.join(data_dir, 'test_daily.csv'))

for df in [train_df, val_df, test_df]:
    df['date'] = pd.to_datetime(df['date'])

train_series = train_df.sort_values('date').set_index('date')[TARGET_DRUG].asfreq('D')
val_series   = val_df.sort_values('date').set_index('date')[TARGET_DRUG].asfreq('D')
test_series  = test_df.sort_values('date').set_index('date')[TARGET_DRUG].asfreq('D')

combined_series = pd.concat([train_series, val_series]).asfreq('D')
full_series     = pd.concat([combined_series, test_series]).asfreq('D')

def evaluate_metrics(y_true, y_pred):
    y_true = np.array(y_true, dtype=float)
    y_pred = np.clip(np.array(y_pred, dtype=float), 0, None)
    rmse  = np.sqrt(np.mean((y_true - y_pred)**2))
    mae   = np.mean(np.abs(y_true - y_pred))
    wape  = np.sum(np.abs(y_true - y_pred)) / np.sum(y_true) * 100
    rmsle = np.sqrt(np.mean((np.log1p(y_true) - np.log1p(y_pred))**2))
    return {'RMSLE': rmsle, 'RMSE': rmse, 'MAE': mae, 'WAPE (%)': wape}

print(f"Dataset for {TARGET_DRUG} loaded successfully!")
print(f"  * Train  : {train_series.index.min().strftime('%Y-%m-%d')} to {train_series.index.max().strftime('%Y-%m-%d')} ({len(train_series)} days)")
print(f"  * Val    : {val_series.index.min().strftime('%Y-%m-%d')} to {val_series.index.max().strftime('%Y-%m-%d')} ({len(val_series)} days)")
print(f"  * Test   : {test_series.index.min().strftime('%Y-%m-%d')} to {test_series.index.max().strftime('%Y-%m-%d')} ({len(test_series)} days)")


Dataset for M01AE loaded successfully!
  * Train  : 2014-01-02 to 2017-12-31 (1460 days)
  * Val    : 2018-01-01 to 2018-12-31 (365 days)
  * Test   : 2019-01-01 to 2019-10-08 (281 days)


In [2]:
# Step 1: PyTorch LSTM Setup & Model Training Code
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

torch.manual_seed(42)
np.random.seed(42)

scaler_mean = combined_series.mean()
scaler_std  = combined_series.std()

scaled_combined = (combined_series - scaler_mean) / scaler_std
seq_length = 28

def create_sequences(series_vals, seq_len):
    xs, ys = [], []
    for i in range(len(series_vals) - seq_len):
        xs.append(series_vals[i:i+seq_len])
        ys.append(series_vals[i+seq_len])
    return np.array(xs), np.array(ys)

X_train_seq, y_train_seq = create_sequences(scaled_combined.values, seq_length)
X_train_t = torch.tensor(X_train_seq, dtype=torch.float32).unsqueeze(-1)
y_train_t = torch.tensor(y_train_seq, dtype=torch.float32).unsqueeze(-1)

class TimeSeriesDataset(Dataset):
    def __init__(self, x, y):
        self.x = x
        self.y = y
    def __len__(self):
        return len(self.x)
    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]

train_loader = DataLoader(TimeSeriesDataset(X_train_t, y_train_t), batch_size=32, shuffle=True)

class LSTMModel(nn.Module):
    def __init__(self, input_size=1, hidden_size=64, num_layers=2, dropout=0.2):
        super(LSTMModel, self).__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True, dropout=dropout)
        self.fc = nn.Linear(hidden_size, 1)
    def forward(self, x):
        out, _ = self.lstm(x)
        out = self.fc(out[:, -1, :])
        return out

model = LSTMModel()
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.005)

model.train()
for epoch in range(40):
    for bx, by in train_loader:
        optimizer.zero_grad()
        out = model(bx)
        loss = criterion(out, by)
        loss.backward()
        optimizer.step()

# Forecasting Test Set
model.eval()
test_inputs = list(scaled_combined.values[-seq_length:])
test_preds_scaled = []

for i in range(len(test_series)):
    x_in = torch.tensor(np.array(test_inputs[-seq_length:]), dtype=torch.float32).view(1, seq_length, 1)
    with torch.no_grad():
        p_val = model(x_in).item()
    test_preds_scaled.append(p_val)
    test_preds_scaled_val = (test_series.iloc[i] - scaler_mean) / scaler_std
    test_inputs.append(test_preds_scaled_val)

m5_test_pred = np.clip(np.array(test_preds_scaled) * scaler_std + scaler_mean, 0, None)

test_metrics = evaluate_metrics(test_series, m5_test_pred)
print(f"=== FINAL TEST HOLD-OUT METRICS (2019) — MODEL 5: PYTORCH LSTM ===")
for k, v in test_metrics.items():
    print(f"  * {k:10s}: {v:.4f}")

pd.DataFrame({'date': test_series.index, 'pred_LSTM': m5_test_pred}).to_csv('m5_lstm_preds.csv', index=False)


=== FINAL TEST HOLD-OUT METRICS (2019) — MODEL 5: PYTORCH LSTM ===
  * RMSLE     : 0.5267
  * RMSE      : 2.3213
  * MAE       : 1.7557
  * WAPE (%)  : 45.4895
